In [1]:
import os
import ast
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler, RobustScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from sklearn.model_selection import GridSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)

from sklearn.naive_bayes import GaussianNB

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

In [4]:
drive_root = "/content/drive/MyDrive"

pos_matches = []
for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_pos_beam_train.csv",
        "scenario23_pos_beam_val.csv",
        "scenario23_pos_beam_test.csv"
    }
    if needed.issubset(set(files)):
        pos_matches.append(root)

print("Position CSV folder candidates:")
for p in pos_matches:
    print(p)

Position CSV folder candidates:
/content/drive/MyDrive/Pos beam


In [5]:
POS_ROOT = "/content/drive/MyDrive/Pos beam"

pos_train_csv = os.path.join(POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(POS_ROOT, "scenario23_pos_beam_test.csv")

print(os.path.exists(pos_train_csv), pos_train_csv)
print(os.path.exists(pos_val_csv), pos_val_csv)
print(os.path.exists(pos_test_csv), pos_test_csv)

True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_train.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_val.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_test.csv


In [6]:
train_df = pd.read_csv(pos_train_csv)
val_df   = pd.read_csv(pos_val_csv)
test_df  = pd.read_csv(pos_test_csv)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

display(train_df.head())
print(train_df.columns.tolist())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)


,index,unit2_pos,unit1_beam
0,3532,"[0.8092883966431671, 0.521083920903955]",17
1,2224,"[0.4816276084988933, 0.29434536152734486]",14
2,9416,"[0.220278556834608, 0.4136596156292844]",17
3,8510,"[0.21412273613497904, 0.4547214157104936]",20
4,6877,"[0.14500641727379412, 0.4097884695072434]",17


['index', 'unit2_pos', 'unit1_beam']


In [7]:
def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value

def add_position_features(df):
    df = df.copy()
    parsed = df["unit2_pos"].apply(parse_unit2_pos)

    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    eps = 1e-8

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["distance2"] = df["distance"] ** 2
    df["distance3"] = df["distance"] ** 3

    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_x3"] = df["pos_x"] ** 3
    df["pos_y3"] = df["pos_y"] ** 3
    df["pos_xy"] = df["pos_x"] * df["pos_y"]

    df["unit_x"] = df["pos_x"] / (df["distance"] + eps)
    df["unit_y"] = df["pos_y"] / (df["distance"] + eps)

    # Extra nonlinear interaction features
    df["sin2_angle"] = np.sin(2 * df["angle"])
    df["cos2_angle"] = np.cos(2 * df["angle"])
    df["sin3_angle"] = np.sin(3 * df["angle"])
    df["cos3_angle"] = np.cos(3 * df["angle"])

    df["dist_sin"] = df["distance"] * df["sin_angle"]
    df["dist_cos"] = df["distance"] * df["cos_angle"]

    return df

train_df_fe = add_position_features(train_df)
val_df_fe   = add_position_features(val_df)
test_df_fe  = add_position_features(test_df)

feature_cols = [
    "pos_x",
    "pos_y",
    "distance",
    "distance2",
    "distance3",
    "angle",
    "sin_angle",
    "cos_angle",
    "pos_x2",
    "pos_y2",
    "pos_x3",
    "pos_y3",
    "pos_xy",
    "unit_x",
    "unit_y",
    "sin2_angle",
    "cos2_angle",
    "sin3_angle",
    "cos3_angle",
    "dist_sin",
    "dist_cos",
]

label_col = "unit1_beam"

print("Number of features:", len(feature_cols))
display(train_df_fe[["index", "unit2_pos", *feature_cols, label_col]].head())
print("Label min/max:", train_df_fe[label_col].min(), train_df_fe[label_col].max())
print("Unique labels:", sorted(train_df_fe[label_col].unique()))

Number of features: 21


,index,unit2_pos,pos_x,pos_y,distance,distance2,distance3,angle,sin_angle,cos_angle,...,pos_xy,unit_x,unit_y,sin2_angle,cos2_angle,sin3_angle,cos3_angle,dist_sin,dist_cos,unit1_beam
0,3532,"[0.8092883966431671, 0.521083920903955]",0.809288,0.521084,0.962536,0.926476,0.891767,0.572060,0.541365,0.840787,...,0.421707,0.840787,0.541365,0.910347,0.413847,0.989450,-0.144873,0.521084,0.809288,17
1,2224,"[0.4816276084988933, 0.29434536152734486]",0.481628,0.294345,0.564450,0.318604,0.179836,0.548576,0.521472,0.853268,...,0.141765,0.853268,0.521472,0.889912,0.456133,0.997194,-0.074861,0.294345,0.481628,14
2,9416,"[0.220278556834608, 0.4136596156292844]",0.220279,0.413660,0.468654,0.219637,0.102934,1.081479,0.882654,0.470023,...,0.091120,0.470023,0.882654,0.829736,-0.558156,-0.102663,-0.994716,0.413660,0.220279,17
3,8510,"[0.21412273613497904, 0.4547214157104936]",0.214123,0.454721,0.502613,0.252620,0.126970,1.130709,0.904714,0.426019,...,0.097366,0.426019,0.904714,0.770851,-0.637016,-0.247920,-0.968780,0.454721,0.214123,20
4,6877,"[0.14500641727379412, 0.4097884695072434]",0.145006,0.409788,0.434688,0.188953,0.082136,1.230690,0.942719,0.333588,...,0.059422,0.333588,0.942719,0.628959,-0.777439,-0.523094,-0.852275,0.409788,0.145006,17


Label min/max: 2 30
Unique labels: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30)]


In [8]:
all_labels = sorted(train_df_fe[label_col].astype(int).unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes = len(all_labels)

print("num_classes:", num_classes)
print("label_to_id:", label_to_id)

num_classes: 29
label_to_id: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3, np.int64(6): 4, np.int64(7): 5, np.int64(8): 6, np.int64(9): 7, np.int64(10): 8, np.int64(11): 9, np.int64(12): 10, np.int64(13): 11, np.int64(14): 12, np.int64(15): 13, np.int64(16): 14, np.int64(17): 15, np.int64(18): 16, np.int64(19): 17, np.int64(20): 18, np.int64(21): 19, np.int64(22): 20, np.int64(23): 21, np.int64(24): 22, np.int64(25): 23, np.int64(26): 24, np.int64(27): 25, np.int64(28): 26, np.int64(29): 27, np.int64(30): 28}


In [9]:
X_train = train_df_fe[feature_cols].astype(float).values
X_val   = val_df_fe[feature_cols].astype(float).values
X_test  = test_df_fe[feature_cols].astype(float).values

y_train_raw = train_df_fe[label_col].astype(int).values
y_val_raw   = val_df_fe[label_col].astype(int).values
y_test_raw  = test_df_fe[label_col].astype(int).values

y_train = np.array([label_to_id[int(y)] for y in y_train_raw])
y_val   = np.array([label_to_id[int(y)] for y in y_val_raw])
y_test  = np.array([label_to_id[int(y)] for y in y_test_raw])

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("Classes:", num_classes)
print("y min/max:", y_train.min(), y_train.max())

X_train: (6832, 21)
X_val  : (3416, 21)
X_test : (1139, 21)
y_train: (6832,)
Classes: 29
y min/max: 0 28


In [10]:
print("Feature columns:", feature_cols)
print("First X row:", X_train[0])
print("First y mapped:", y_train[0])
print("First y original:", id_to_label[y_train[0]])

assert X_train.shape[1] == len(feature_cols)
assert y_train.min() >= 0
assert y_train.max() < num_classes

print("Data sanity check passed.")

Feature columns: ['pos_x', 'pos_y', 'distance', 'distance2', 'distance3', 'angle', 'sin_angle', 'cos_angle', 'pos_x2', 'pos_y2', 'pos_x3', 'pos_y3', 'pos_xy', 'unit_x', 'unit_y', 'sin2_angle', 'cos2_angle', 'sin3_angle', 'cos3_angle', 'dist_sin', 'dist_cos']
First X row: [ 0.8092884   0.52108392  0.96253632  0.92647616  0.89176695  0.57206029
  0.54136547  0.84078739  0.65494771  0.27152845  0.53004158  0.14148911
  0.42170717  0.84078738  0.54136546  0.91034651  0.41384687  0.98945027
 -0.14487294  0.52108392  0.8092884 ]
First y mapped: 15
First y original: 17
Data sanity check passed.


In [11]:
def get_proba_safe(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        exp_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
        return exp_scores / exp_scores.sum(axis=1, keepdims=True)

    raise ValueError("Model has neither predict_proba nor decision_function.")

def evaluate_ml_model(model, X, y, ks=(1, 2, 3, 5)):
    proba = get_proba_safe(model, X)

    results = {}
    pred = np.argmax(proba, axis=1)
    results["top1"] = accuracy_score(y, pred) * 100

    for k in ks:
        results[f"top{k}"] = top_k_accuracy_score(
            y,
            proba,
            k=k,
            labels=np.arange(proba.shape[1])
        ) * 100

    return results

In [12]:
ml_models = {
    "LogisticRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            C=3.0,
            solver="lbfgs",
            multi_class="auto",
            n_jobs=-1,
            random_state=SEED
        ))
    ]),

    "KNN_distance": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(
            n_neighbors=9,
            weights="distance",
            metric="minkowski",
            p=2
        ))
    ]),

    "SVM_RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C=20,
            gamma="scale",
            kernel="rbf",
            probability=True,
            random_state=SEED
        ))
    ]),

    "RandomForest": RandomForestClassifier(
        n_estimators=800,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features="sqrt",
        bootstrap=True,
        random_state=SEED,
        n_jobs=-1
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=1200,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features=None,
        bootstrap=False,
        random_state=SEED,
        n_jobs=-1
    ),

    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=400,
        learning_rate=0.04,
        max_depth=3,
        subsample=0.9,
        random_state=SEED
    ),

    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=500,
        learning_rate=0.04,
        max_leaf_nodes=31,
        l2_regularization=1e-4,
        random_state=SEED
    ),
}

ml_results = []
trained_models = {}

for name, model in ml_models.items():
    print("\n" + "="*80)
    print("Training:", name)
    print("="*80)

    model.fit(X_train, y_train)
    trained_models[name] = model

    val_metrics = evaluate_ml_model(model, X_val, y_val)
    test_metrics = evaluate_ml_model(model, X_test, y_test)

    print("Val :", val_metrics)
    print("Test:", test_metrics)

    ml_results.append({
        "Model": name,
        "Val Top-1": val_metrics["top1"],
        "Val Top-2": val_metrics["top2"],
        "Val Top-3": val_metrics["top3"],
        "Val Top-5": val_metrics["top5"],
        "Test Top-1": test_metrics["top1"],
        "Test Top-2": test_metrics["top2"],
        "Test Top-3": test_metrics["top3"],
        "Test Top-5": test_metrics["top5"],
    })

ml_results_df = pd.DataFrame(ml_results).sort_values(by="Test Top-1", ascending=False).reset_index(drop=True)
ml_results_df


Training: LogisticRegression
Val : {'top1': np.float64(55.26932084309133), 'top2': np.float64(77.72248243559719), 'top3': np.float64(87.67564402810304), 'top5': np.float64(95.90163934426229)}
Test: {'top1': np.float64(52.06321334503951), 'top2': np.float64(76.29499561018437), 'top3': np.float64(87.35733099209834), 'top5': np.float64(96.22475856014047)}

Training: KNN_distance
Val : {'top1': np.float64(67.97423887587823), 'top2': np.float64(84.4847775175644), 'top3': np.float64(90.1639344262295), 'top5': np.float64(92.88641686182669)}
Test: {'top1': np.float64(67.60316066725197), 'top2': np.float64(83.14310798946444), 'top3': np.float64(90.07901668129938), 'top5': np.float64(92.09833187006146)}

Training: SVM_RBF
Val : {'top1': np.float64(61.85597189695551), 'top2': np.float64(82.69906323185012), 'top3': np.float64(91.18852459016394), 'top5': np.float64(97.8337236533958)}
Test: {'top1': np.float64(57.50658472344161), 'top2': np.float64(81.82616330114135), 'top3': np.float64(91.74714661

,Model,Val Top-1,Val Top-2,Val Top-3,Val Top-5,Test Top-1,Test Top-2,Test Top-3,Test Top-5
0,RandomForest,69.203747,85.597190,92.798595,97.277518,68.744513,86.742757,93.678665,97.366111
1,KNN_distance,67.974239,84.484778,90.163934,92.886417,67.603161,83.143108,90.079017,92.098332
2,GradientBoosting,67.535129,85.040984,92.008197,96.692037,66.812994,85.513608,91.834943,96.927129
3,ExtraTrees,67.125293,84.426230,92.066745,96.194379,66.725198,84.723442,92.712906,95.873573
4,HistGradientBoosting,67.418033,85.158080,92.798595,96.721311,66.374012,84.986831,92.888499,97.629500
5,SVM_RBF,61.855972,82.699063,91.188525,97.833724,57.506585,81.826163,91.747147,97.980685
6,LogisticRegression,55.269321,77.722482,87.675644,95.901639,52.063213,76.294996,87.357331,96.224759


In [13]:
try:
    from xgboost import XGBClassifier

    xgb_model = XGBClassifier(
        n_estimators=800,
        max_depth=4,
        learning_rate=0.025,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        reg_alpha=0.0,
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        random_state=SEED,
        n_jobs=-1
    )

    print("Training XGBoost...")
    xgb_model.fit(X_train, y_train)

    xgb_val_metrics = evaluate_ml_model(xgb_model, X_val, y_val)
    xgb_test_metrics = evaluate_ml_model(xgb_model, X_test, y_test)

    print("XGB Val :", xgb_val_metrics)
    print("XGB Test:", xgb_test_metrics)

    trained_models["XGBoost"] = xgb_model

    ml_results_df = pd.concat([
        ml_results_df,
        pd.DataFrame([{
            "Model": "XGBoost",
            "Val Top-1": xgb_val_metrics["top1"],
            "Val Top-2": xgb_val_metrics["top2"],
            "Val Top-3": xgb_val_metrics["top3"],
            "Val Top-5": xgb_val_metrics["top5"],
            "Test Top-1": xgb_test_metrics["top1"],
            "Test Top-2": xgb_test_metrics["top2"],
            "Test Top-3": xgb_test_metrics["top3"],
            "Test Top-5": xgb_test_metrics["top5"],
        }])
    ], ignore_index=True)

    ml_results_df = ml_results_df.sort_values(by="Test Top-1", ascending=False).reset_index(drop=True)

except Exception as e:
    print("XGBoost failed:", e)

ml_results_df

Training XGBoost...
XGB Val : {'top1': np.float64(69.40866510538642), 'top2': np.float64(86.82669789227167), 'top3': np.float64(93.61826697892272), 'top5': np.float64(98.06791569086651)}
XGB Test: {'top1': np.float64(68.74451273046532), 'top2': np.float64(88.14749780509219), 'top3': np.float64(93.94205443371378), 'top5': np.float64(97.98068481123792)}


,Model,Val Top-1,Val Top-2,Val Top-3,Val Top-5,Test Top-1,Test Top-2,Test Top-3,Test Top-5
0,RandomForest,69.203747,85.597190,92.798595,97.277518,68.744513,86.742757,93.678665,97.366111
1,XGBoost,69.408665,86.826698,93.618267,98.067916,68.744513,88.147498,93.942054,97.980685
2,KNN_distance,67.974239,84.484778,90.163934,92.886417,67.603161,83.143108,90.079017,92.098332
3,GradientBoosting,67.535129,85.040984,92.008197,96.692037,66.812994,85.513608,91.834943,96.927129
4,ExtraTrees,67.125293,84.426230,92.066745,96.194379,66.725198,84.723442,92.712906,95.873573
5,HistGradientBoosting,67.418033,85.158080,92.798595,96.721311,66.374012,84.986831,92.888499,97.629500
6,SVM_RBF,61.855972,82.699063,91.188525,97.833724,57.506585,81.826163,91.747147,97.980685
7,LogisticRegression,55.269321,77.722482,87.675644,95.901639,52.063213,76.294996,87.357331,96.224759


In [14]:
try:
    from lightgbm import LGBMClassifier

    lgbm_model = LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.025,
        max_depth=-1,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multiclass",
        random_state=SEED,
        n_jobs=-1
    )

    print("Training LightGBM...")
    lgbm_model.fit(X_train, y_train)

    lgbm_val_metrics = evaluate_ml_model(lgbm_model, X_val, y_val)
    lgbm_test_metrics = evaluate_ml_model(lgbm_model, X_test, y_test)

    print("LGBM Val :", lgbm_val_metrics)
    print("LGBM Test:", lgbm_test_metrics)

    trained_models["LightGBM"] = lgbm_model

    ml_results_df = pd.concat([
        ml_results_df,
        pd.DataFrame([{
            "Model": "LightGBM",
            "Val Top-1": lgbm_val_metrics["top1"],
            "Val Top-2": lgbm_val_metrics["top2"],
            "Val Top-3": lgbm_val_metrics["top3"],
            "Val Top-5": lgbm_test_metrics["top5"],
            "Test Top-1": lgbm_test_metrics["top1"],
            "Test Top-2": lgbm_test_metrics["top2"],
            "Test Top-3": lgbm_test_metrics["top3"],
            "Test Top-5": lgbm_test_metrics["top5"],
        }])
    ], ignore_index=True)

    ml_results_df = ml_results_df.sort_values(by="Test Top-1", ascending=False).reset_index(drop=True)

except Exception as e:
    print("LightGBM failed:", e)

ml_results_df

Training LightGBM...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001710 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 6832, number of used features: 21
[LightGBM] [Info] Start training from score -3.561515
[LightGBM] [Info] Start training from score -4.937552
[LightGBM] [Info] Start training from score -3.582349
[LightGBM] [Info] Start training from score -6.431477
[LightGBM] [Info] Start training from score -4.654985
[LightGBM] [Info] Start training from score -4.363465
[LightGBM] [Info] Start training from score -5.165811
[LightGBM] [Info] Start training from score -3.647589
[LightGBM] [Info] Start training from score -4.840389
[LightGBM] [Info] Start training from score -3.253424
[LightGBM] [Info] Start training from score -2.988731
[LightGBM] [Info] Start training from score -4.214252
[LightGBM] [Info] Start training from score

,Model,Val Top-1,Val Top-2,Val Top-3,Val Top-5,Test Top-1,Test Top-2,Test Top-3,Test Top-5
0,RandomForest,69.203747,85.597190,92.798595,97.277518,68.744513,86.742757,93.678665,97.366111
1,XGBoost,69.408665,86.826698,93.618267,98.067916,68.744513,88.147498,93.942054,97.980685
2,KNN_distance,67.974239,84.484778,90.163934,92.886417,67.603161,83.143108,90.079017,92.098332
3,GradientBoosting,67.535129,85.040984,92.008197,96.692037,66.812994,85.513608,91.834943,96.927129
4,ExtraTrees,67.125293,84.426230,92.066745,96.194379,66.725198,84.723442,92.712906,95.873573
5,HistGradientBoosting,67.418033,85.158080,92.798595,96.721311,66.374012,84.986831,92.888499,97.629500
6,LightGBM,67.359485,85.158080,92.125293,97.366111,66.286216,85.250219,92.537313,97.366111
7,SVM_RBF,61.855972,82.699063,91.188525,97.833724,57.506585,81.826163,91.747147,97.980685
8,LogisticRegression,55.269321,77.722482,87.675644,95.901639,52.063213,76.294996,87.357331,96.224759


In [15]:
ml_results_df = ml_results_df.sort_values(by="Val Top-1", ascending=False).reset_index(drop=True)
display(ml_results_df)

top_model_names = ml_results_df["Model"].head(4).tolist()
print("Selected models for ensemble:", top_model_names)

,Model,Val Top-1,Val Top-2,Val Top-3,Val Top-5,Test Top-1,Test Top-2,Test Top-3,Test Top-5
0,XGBoost,69.408665,86.826698,93.618267,98.067916,68.744513,88.147498,93.942054,97.980685
1,RandomForest,69.203747,85.597190,92.798595,97.277518,68.744513,86.742757,93.678665,97.366111
2,KNN_distance,67.974239,84.484778,90.163934,92.886417,67.603161,83.143108,90.079017,92.098332
3,GradientBoosting,67.535129,85.040984,92.008197,96.692037,66.812994,85.513608,91.834943,96.927129
4,HistGradientBoosting,67.418033,85.158080,92.798595,96.721311,66.374012,84.986831,92.888499,97.629500
5,LightGBM,67.359485,85.158080,92.125293,97.366111,66.286216,85.250219,92.537313,97.366111
6,ExtraTrees,67.125293,84.426230,92.066745,96.194379,66.725198,84.723442,92.712906,95.873573
7,SVM_RBF,61.855972,82.699063,91.188525,97.833724,57.506585,81.826163,91.747147,97.980685
8,LogisticRegression,55.269321,77.722482,87.675644,95.901639,52.063213,76.294996,87.357331,96.224759


Selected models for ensemble: ['XGBoost', 'RandomForest', 'KNN_distance', 'GradientBoosting']


In [16]:
voting_estimators = []

for name in top_model_names:
    if name in trained_models:
        voting_estimators.append((name, trained_models[name]))

print("Voting estimators:", [n for n, _ in voting_estimators])

soft_voting = VotingClassifier(
    estimators=voting_estimators,
    voting="soft",
    n_jobs=-1
)

soft_voting.fit(X_train, y_train)

voting_val_metrics = evaluate_ml_model(soft_voting, X_val, y_val)
voting_test_metrics = evaluate_ml_model(soft_voting, X_test, y_test)

print("Soft Voting Val :", voting_val_metrics)
print("Soft Voting Test:", voting_test_metrics)

Voting estimators: ['XGBoost', 'RandomForest', 'KNN_distance', 'GradientBoosting']
Soft Voting Val : {'top1': np.float64(69.49648711943794), 'top2': np.float64(86.82669789227167), 'top3': np.float64(93.61826697892272), 'top5': np.float64(97.95081967213115)}
Soft Voting Test: {'top1': np.float64(70.06145741878841), 'top2': np.float64(87.62071992976294), 'top3': np.float64(93.76646180860403), 'top5': np.float64(98.15627743634766)}


Manual Probability Averaging Ensemble

In [17]:
def evaluate_manual_proba_ensemble(models_dict, model_names, X, y, ks=(1, 2, 3, 5), weights=None):
    probas = []

    if weights is None:
        weights = np.ones(len(model_names)) / len(model_names)
    else:
        weights = np.array(weights)
        weights = weights / weights.sum()

    for name in model_names:
        model = models_dict[name]
        proba = get_proba_safe(model, X)
        probas.append(proba)

    avg_proba = np.zeros_like(probas[0])
    for w, p in zip(weights, probas):
        avg_proba += w * p

    pred = np.argmax(avg_proba, axis=1)

    results = {"top1": accuracy_score(y, pred) * 100}
    for k in ks:
        results[f"top{k}"] = top_k_accuracy_score(
            y,
            avg_proba,
            k=k,
            labels=np.arange(avg_proba.shape[1])
        ) * 100

    return results

manual_val_metrics = evaluate_manual_proba_ensemble(
    trained_models,
    top_model_names,
    X_val,
    y_val
)

manual_test_metrics = evaluate_manual_proba_ensemble(
    trained_models,
    top_model_names,
    X_test,
    y_test
)

print("Manual Ensemble Val :", manual_val_metrics)
print("Manual Ensemble Test:", manual_test_metrics)

Manual Ensemble Val : {'top1': np.float64(69.49648711943794), 'top2': np.float64(86.82669789227167), 'top3': np.float64(93.61826697892272), 'top5': np.float64(97.95081967213115)}
Manual Ensemble Test: {'top1': np.float64(70.06145741878841), 'top2': np.float64(87.62071992976294), 'top3': np.float64(93.76646180860403), 'top5': np.float64(98.15627743634766)}


In [18]:
def ensemble_predict_proba(models_dict, model_names, X, weights):
    probas = [get_proba_safe(models_dict[name], X) for name in model_names]
    weights = np.array(weights, dtype=float)
    weights = weights / weights.sum()

    avg_proba = np.zeros_like(probas[0])
    for w, p in zip(weights, probas):
        avg_proba += w * p

    return avg_proba

def score_weights(models_dict, model_names, X, y, weights):
    proba = ensemble_predict_proba(models_dict, model_names, X, weights)
    pred = np.argmax(proba, axis=1)
    return accuracy_score(y, pred) * 100

rng = np.random.default_rng(SEED)

best_weight_score = -1
best_weights = None

n_search = 1000

for _ in range(n_search):
    weights = rng.dirichlet(np.ones(len(top_model_names)))
    score = score_weights(trained_models, top_model_names, X_val, y_val, weights)

    if score > best_weight_score:
        best_weight_score = score
        best_weights = weights

print("Best validation weighted ensemble score:", best_weight_score)
print("Best weights:")
for name, w in zip(top_model_names, best_weights):
    print(f"{name}: {w:.4f}")

Best validation weighted ensemble score: 70.43325526932084
Best weights:
XGBoost: 0.0059
RandomForest: 0.4739
KNN_distance: 0.3666
GradientBoosting: 0.1535


In [ ]:
stack_estimators = []

for name in top_model_names:
    if name in trained_models:
        stack_estimators.append((name, trained_models[name]))

stacking_model = StackingClassifier(
    estimators=stack_estimators,
    final_estimator=LogisticRegression(
        max_iter=2000,
        C=1.0,
        multi_class="auto",
        n_jobs=-1
    ),
    stack_method="predict_proba",
    passthrough=True,
    n_jobs=-1,
    cv=5
)

print("Training StackingClassifier...")
stacking_model.fit(X_train, y_train)

stack_val_metrics = evaluate_ml_model(stacking_model, X_val, y_val)
stack_test_metrics = evaluate_ml_model(stacking_model, X_test, y_test)

print("Stacking Val :", stack_val_metrics)
print("Stacking Test:", stack_test_metrics)

Training StackingClassifier...


In [ ]:
final_rows = []


for _, row in ml_results_df.iterrows():
    final_rows.append({
        "Model": row["Model"],
        "Val Top-1": row["Val Top-1"],
        "Val Top-2": row["Val Top-2"],
        "Val Top-3": row["Val Top-3"],
        "Val Top-5": row["Val Top-5"],
        "Test Top-1": row["Test Top-1"],
        "Test Top-2": row["Test Top-2"],
        "Test Top-3": row["Test Top-3"],
        "Test Top-5": row["Test Top-5"],
    })

# Add ensembles
final_rows.extend([
    {
        "Model": "SoftVoting",
        "Val Top-1": voting_val_metrics["top1"],
        "Val Top-2": voting_val_metrics["top2"],
        "Val Top-3": voting_val_metrics["top3"],
        "Val Top-5": voting_val_metrics["top5"],
        "Test Top-1": voting_test_metrics["top1"],
        "Test Top-2": voting_test_metrics["top2"],
        "Test Top-3": voting_test_metrics["top3"],
        "Test Top-5": voting_test_metrics["top5"],
    },
    {
        "Model": "ManualAvgVoting",
        "Val Top-1": manual_val_metrics["top1"],
        "Val Top-2": manual_val_metrics["top2"],
        "Val Top-3": manual_val_metrics["top3"],
        "Val Top-5": manual_val_metrics["top5"],
        "Test Top-1": manual_test_metrics["top1"],
        "Test Top-2": manual_test_metrics["top2"],
        "Test Top-3": manual_test_metrics["top3"],
        "Test Top-5": manual_test_metrics["top5"],
    },
    {
        "Model": "WeightedVoting",
        "Val Top-1": weighted_val_metrics["top1"],
        "Val Top-2": weighted_val_metrics["top2"],
        "Val Top-3": weighted_val_metrics["top3"],
        "Val Top-5": weighted_val_metrics["top5"],
        "Test Top-1": weighted_test_metrics["top1"],
        "Test Top-2": weighted_test_metrics["top2"],
        "Test Top-3": weighted_test_metrics["top3"],
        "Test Top-5": weighted_test_metrics["top5"],
    },
    {
        "Model": "Stacking",
        "Val Top-1": stack_val_metrics["top1"],
        "Val Top-2": stack_val_metrics["top2"],
        "Val Top-3": stack_val_metrics["top3"],
        "Val Top-5": stack_val_metrics["top5"],
        "Test Top-1": stack_test_metrics["top1"],
        "Test Top-2": stack_test_metrics["top2"],
        "Test Top-3": stack_test_metrics["top3"],
        "Test Top-5": stack_test_metrics["top5"],
    },
])

final_results_df = pd.DataFrame(final_rows)
final_results_df = final_results_df.sort_values(by="Test Top-1", ascending=False).reset_index(drop=True)
final_results_df

In [ ]:
plot_df = final_results_df.head(10).set_index("Model")

plt.figure(figsize=(11, 5))
plt.bar(plot_df.index, plot_df["Test Top-1"])
plt.ylabel("Test Top-1 Accuracy (%)")
plt.title("Traditional ML + Ensemble Position-Aided Beam Prediction")
plt.xticks(rotation=30)
plt.grid(True, axis="y")
plt.show()

plt.figure(figsize=(11, 5))
plt.bar(plot_df.index, plot_df["Test Top-5"])
plt.ylabel("Test Top-5 Accuracy (%)")
plt.title("Traditional ML + Ensemble Top-5 Comparison")
plt.xticks(rotation=30)
plt.grid(True, axis="y")
plt.show()